# 본문 BS4 실패 URL 검수 (로컬 + Selenium)

`본문_bs4_재실패_*.json` 의 실패 URL을 1건씩 Selenium으로 띄워 사람이 눈으로 보고 라벨링하는 노트북

- 입력 — `data/news/본문_bs4_재실패_*.json` 전부
- 동작 — URL 하나씩 띄움 → 너가 보고 Enter(속보) / `a`(anomaly + 메모) / `s`(skip) / `q`(중단·저장 후 종료)
- 저장 — 매 라벨 즉시 CSV append (도중 끊겨도 손실 0, 다시 실행하면 이미 라벨된 link skip)
- 산출 `outputs/` — `실패검수_라벨.csv`(전건) · `실패검수_press별요약.csv`(press×label 표) · `실패검수_anomaly.csv`(anomaly만)
- 발표용 결론은 마지막 셀이 자동 생성


In [1]:
# 한 번만 — selenium 없으면 설치
# !pip install -q selenium


In [2]:
from pathlib import Path
import os, re, json, time, unicodedata, datetime as dt
import pandas as pd

PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')
DATA_DIR = PROJECT_DIR / 'data' / 'news'
OUT_DIR  = PROJECT_DIR / 'outputs'; OUT_DIR.mkdir(exist_ok=True)
LABEL_CSV = OUT_DIR / '실패검수_라벨.csv'

def nn(p): return unicodedata.normalize('NFC', p.name)
print('DATA_DIR :', DATA_DIR, '/ 존재', DATA_DIR.exists())
print('OUT_DIR  :', OUT_DIR)
print('LABEL CSV:', LABEL_CSV, '/ 이미 있음' if LABEL_CSV.exists() else '/ 새로 만듦')


DATA_DIR : /home/carol/Text-data-Analysis_26-Spring/data/news / 존재 True
OUT_DIR  : /home/carol/Text-data-Analysis_26-Spring/outputs
LABEL CSV: /home/carol/Text-data-Analysis_26-Spring/outputs/실패검수_라벨.csv / 새로 만듦


In [3]:
# --- 실패 파일 글러빙 + 작업 목록 만들기 ---
records = []
pat = re.compile(r'^본문_bs4_재실패_(.+?)_(\d{6})_(\d{6})\.json$')
for fp in sorted(DATA_DIR.iterdir() if DATA_DIR.exists() else []):
    m = pat.match(nn(fp))
    if not m: continue
    press, s, e = m.groups()
    data = json.loads(fp.read_text(encoding='utf-8'))
    links = data.get('links', [])
    for link in links:
        records.append({'press': press, 'period': f'{s}_{e}', 'link': link})
    print(f'  {press} {s}_{e}: {len(links)}건')
print(f'\n전체 실패 URL: {len(records)}건')


  매일경제 260505_260511: 26건
  한겨레 260505_260511: 1건
  한국경제 260505_260511: 2건

전체 실패 URL: 29건


In [4]:
# --- 이미 라벨링된 link 빼고 남은 작업 산출 ---
if LABEL_CSV.exists():
    done_df = pd.read_csv(LABEL_CSV, encoding='utf-8-sig')
    done_set = set(done_df['link'].tolist())
    print(f'이미 라벨: {len(done_set)}건')
else:
    done_set = set()
    print('새 시작 — 0건 라벨됨')

pending = [r for r in records if r['link'] not in done_set]
print(f'이번 세션 작업 대상: {len(pending)}건')
# press별 잔여
from collections import Counter
left_by_press = Counter(r['press'] for r in pending)
for p, n in sorted(left_by_press.items()): print(f'  {p}: {n}건')


새 시작 — 0건 라벨됨
이번 세션 작업 대상: 29건
  매일경제: 26건
  한겨레: 1건
  한국경제: 2건


In [6]:
# --- Selenium driver 띄움(headed — 너가 직접 봐야 함) ---
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
opts = Options()
opts.add_argument('--window-size=1200,900')
opts.add_argument('--lang=ko-KR')
# user-agent 일반 데스크탑으로
opts.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36')
driver = webdriver.Chrome(options=opts)   # Selenium Manager가 chromedriver 자동 fetch
print('Chrome 준비 완료 — 메인 셀로 진행')


Chrome 준비 완료 — 메인 셀로 진행


In [ ]:
# --- 메인 라벨링 루프 ---
# 키: Enter=속보  /  a=anomaly(메모 입력)  /  s=skip  /  q=중단
# 매 라벨 즉시 CSV append → 중단되어도 손실 없음, 다시 실행 시 자동 이어하기
print('Enter=속보 / a=anomaly / s=skip / q=quit')
print('-' * 70)

try:
    for i, r in enumerate(pending, 1):
        print(f'[{i}/{len(pending)}] {r["press"]} {r["period"]}')
        print(f'  link : {r["link"]}')
        try:
            driver.get(r['link'])
            time.sleep(1.5)
            title = driver.title or ''
        except Exception as ex:
            title = f'(load error: {ex})'
        print(f'  title: {title[:100]}')
        inp = input('label> ').strip().lower()
        if inp == 'q':
            print('중단 — 진행 상황 저장됨'); break
        if inp == 's':
            print('  → skip\n'); continue
        if inp == 'a':
            note = input('anomaly note(짧게)> ').strip()
            label = 'anomaly'
        else:
            label = '속보'
            note = ''
        row = {
            'press': r['press'], 'period': r['period'], 'link': r['link'],
            'title': title, 'label': label, 'note': note,
            'timestamp': dt.datetime.now().isoformat(timespec='seconds'),
        }
        new_file = not LABEL_CSV.exists()
        pd.DataFrame([row]).to_csv(LABEL_CSV, mode='a', header=new_file,
                                   index=False, encoding='utf-8-sig')
        print(f'  → {label}{(" ("+note+")") if note else ""}\n')
except KeyboardInterrupt:
    print('\nKeyboard interrupt — 진행 상황 저장됨')
finally:
    try: driver.quit()
    except Exception: pass
print('루프 종료')


Enter=속보 / a=anomaly / s=skip / q=quit
----------------------------------------------------------------------
[1/29] 매일경제 260505_260511
  link : https://n.news.naver.com/mnews/article/009/0005675484
  title: [속보] 청와대 “해수부·청해부대, 사고선박 선원 안전 실시간 파악”
  → 속보

[2/29] 매일경제 260505_260511
  link : https://n.news.naver.com/mnews/article/009/0005675489
  title: "이 강연 놓치면 후회할 것 같아"… 머니쇼 일정 꼭 체크하세요
  → 속보

[3/29] 매일경제 260505_260511
  link : https://n.news.naver.com/mnews/article/009/0005675497
  title: [속보] 청와대 “선사 자체조사 별도로 해양심판원·소방청 인력 급파
  → 속보

[4/29] 매일경제 260505_260511
  link : https://n.news.naver.com/mnews/article/009/0005675541
  title: 카툰포커스
  → 속보

[5/29] 매일경제 260505_260511
  link : https://n.news.naver.com/mnews/article/009/0005675547
  title: 아이디
  → 속보

[6/29] 매일경제 260505_260511
  link : https://n.news.naver.com/mnews/article/009/0005675548
  title: [표] 오늘의 날씨
  → 속보

[7/29] 매일경제 260505_260511
  link : https://n.news.naver.com/mnews/article/009/0005675586
  title: [표] 외국환율고시표
  → 속보

[8

In [8]:

# --- 요약 + 산출 저장 ---
if not LABEL_CSV.exists():
    print('아직 라벨링 0건 — 메인 셀부터 실행하세요'); 
else:
    df = pd.read_csv(LABEL_CSV, encoding='utf-8-sig')
    print(f'총 라벨링: {len(df)}건')
    print('\n=== 전체 라벨 분포 ===')
    print(df['label'].value_counts())
    print('\n=== press × label ===')
    summary = (df.groupby(['press','label']).size()
                 .unstack(fill_value=0)
                 .assign(합계=lambda d: d.sum(axis=1)))
    print(summary)
    summary.to_csv(OUT_DIR / '실패검수_press별요약.csv', encoding='utf-8-sig')
    anomaly_df = df[df['label']=='anomaly'].copy()
    anomaly_df.to_csv(OUT_DIR / '실패검수_anomaly.csv', index=False, encoding='utf-8-sig')
    print(f'\n저장: 실패검수_press별요약.csv ({len(summary)} press) / 실패검수_anomaly.csv ({len(anomaly_df)}건)')
    # 발표용 자동 결론
    n_total = len(df); n_anom = (df['label']=='anomaly').sum()
    pct_anom = (n_anom / n_total * 100) if n_total else 0
    pct_norm = 100 - pct_anom
    print('\n=== 발표용 요약 (한 단락) ===')
    print(f'BS4 본문 수집 실패 {n_total}건을 사람이 직접 검수한 결과, '
          f'{pct_norm:.1f}% ({n_total-n_anom}건)는 사진 위주의 속보로 본문 부재 — 데이터 누락이 아닌 컨텐츠 부재. '
          f'나머지 {pct_anom:.1f}% ({n_anom}건)는 별도 검토가 필요한 anomaly로 분류 — 상세는 outputs/실패검수_anomaly.csv 참조.')


총 라벨링: 29건

=== 전체 라벨 분포 ===
label
속보    29
Name: count, dtype: int64

=== press × label ===
label  속보  합계
press        
매일경제   26  26
한겨레     1   1
한국경제    2   2

저장: 실패검수_press별요약.csv (3 press) / 실패검수_anomaly.csv (0건)

=== 발표용 요약 (한 단락) ===
BS4 본문 수집 실패 29건을 사람이 직접 검수한 결과, 100.0% (29건)는 사진 위주의 속보로 본문 부재 — 데이터 누락이 아닌 컨텐츠 부재. 나머지 0.0% (0건)는 별도 검토가 필요한 anomaly로 분류 — 상세는 outputs/실패검수_anomaly.csv 참조.


## 검증 체크리스트
- 실패 파일 글러빙 결과 press별 건수 표시
- 라벨 즉시 append — 중단 후 재실행 시 done_set 으로 자동 이어하기
- anomaly 발견 시 메모 한 줄 입력 → CSV에 같이 저장
- 최종 요약 — press × label 표·anomaly 별도 CSV·발표용 한 단락 자동 출력
